In [ ]:
subscr_min = (
    subscr[subscr['subscribed_after_view_flg'] == 1]
    .groupby('contact_id', as_index=False)
    .agg(first_subscribe_day=('first_subscr_dt', 'min'))
)

subscr_min['first_subscribe_day'] = pd.to_datetime(
    subscr_min['first_subscribe_day']
)

view_offer = (
    subscr
    .groupby('contact_id', as_index=False)
    .agg(view_offer_flg=('view_offer_flg', 'max'))
)

scores = pd.concat(
    [
        march_scores.assign(score_month='Март'),
        april_scores.assign(score_month='Апрель')
    ],
    ignore_index=True
)

scores['score'] = pd.to_numeric(scores['score'], errors='coerce')

merged_scores = (
    scores
    .merge(subscr_min, on='contact_id', how='left')
    .merge(view_offer, on='contact_id', how='left')
)

merged_scores['is_subscribed'] = (
    merged_scores['first_subscribe_day']
    .notna()
).astype(int)

merged_scores['view_offer_flg'] = (
    merged_scores['view_offer_flg']
    .fillna(0)
    .astype(int)
)

merged_scores['score_bin'] = pd.cut(
    merged_scores['score'],
    bins=6
)

score_stats = (
    merged_scores
    .groupby(['score_month', 'score_bin'], as_index=False, observed=True)
    .agg(
        avg_score=('score', 'mean'),
        clients_cnt=('contact_id', 'nunique'),
        subscribed_cnt=('is_subscribed', 'sum'),
        viewed_offer_cnt=('view_offer_flg', 'sum')
    )
)

score_stats['subscribed_pct'] = (
    score_stats['subscribed_cnt']
    / score_stats['clients_cnt']
    * 100
).round(2)

score_stats['viewed_offer_pct'] = (
    score_stats['viewed_offer_cnt']
    / score_stats['clients_cnt']
    * 100
).round(2)

score_stats['score_bin'] = score_stats['score_bin'].apply(
    lambda x: f'{x.left:.2f} - {x.right:.2f}'
)